# hashrope demo: one data structure, four languages

Companion notebook for the talk **Hashrope: One Data Structure, Four Languages** (DevFest Florida 2026 hackathon), by Muntaser Syed.

hashrope is a rope (a balanced binary tree of byte segments) where every node also carries a polynomial fingerprint of the bytes below it. The root fingerprint identifies the whole sequence, any substring can be fingerprinted in O(log n) time, and concat, split and insert stay logarithmic. The same design ships in Python, JavaScript/TypeScript, Rust and Java at one version, 0.3.4.

This notebook runs the Python package and the npm package side by side. Run it top to bottom (Runtime, Run all). Everything here runs on a free Colab CPU runtime in well under a minute.

Links:
- Live explainer site with a playground: https://hashrope.lovable.app/
- Source and release records: https://github.com/jemsbhai/hashrope
- One-page cheat sheet: https://gist.github.com/jemsbhai (file hashrope-cheatsheet.md)
- Packages: [PyPI](https://pypi.org/project/hashrope/), [npm](https://www.npmjs.com/package/hashrope), [crates.io](https://crates.io/crates/hashrope), [Maven Central](https://central.sonatype.com/artifact/io.github.jemsbhai/hashrope)

## 0. Install

Zero runtime dependencies in every language, so the install is quick.

In [1]:
!pip install -q --root-user-action=ignore "hashrope==0.3.4"
import hashrope
print("hashrope", hashrope.__version__)

hashrope 0.3.4


## 1. Build a rope and read its fingerprint

Bytes live in leaves. An internal node stores the length, the weight (leaf count) and the fingerprint of everything below it, computed once from its two children with

H(A || B) = H(A) * x^|B| + H(B)   (mod p, with p = 2^61 - 1 and x = 131 by default).

So the root fingerprint of the tree equals the fingerprint of the flat byte string, and it never re-reads bytes it already hashed.

In [2]:
from hashrope import PolynomialHash, Leaf, rope_concat, rope_split, rope_hash, rope_len, rope_to_bytes

h = PolynomialHash()                      # defaults: p = 2^61 - 1, base 131
print("profile:", h.profile)

hello = Leaf(b"hello ", h)
world = Leaf(b"world", h)
rope = rope_concat(hello, world, h)       # one new node; the two leaves are shared, not copied

print("rope fingerprint :", hex(rope_hash(rope)))
print("direct fingerprint:", hex(h.hash(b"hello world")))
print("equal:", rope_hash(rope) == h.hash(b"hello world"))
print("length:", rope_len(rope), "bytes")

profile: HashProfile(prime=2305843009213693951, base=131)
rope fingerprint : 0x5f87bb7db23cc7b
direct fingerprint: 0x5f87bb7db23cc7b
equal: True
length: 11 bytes


Remember that value, `0x5f87bb7db23cc7b`. Section 9 prints the same number from JavaScript.

## 2. Split anywhere, fingerprint any substring

All positions are byte offsets (encode text explicitly; a split may land inside a multibyte UTF-8 character and that is still exact at the byte level). `rope_substr_hash` walks O(log n) subtrees and allocates no new rope nodes.

In [3]:
from hashrope import rope_substr_hash

left, right = rope_split(rope, 6, h)
print("left :", rope_to_bytes(left))
print("right:", rope_to_bytes(right))

sub = rope_substr_hash(rope, 6, 5, h)     # bytes 6..10 of the rope, no allocation
print("substring fingerprint matches direct hash of b'world':", sub == h.hash(b"world"))

left : b'hello '
right: b'world'
substring fingerprint matches direct hash of b'world': True


## 3. A 1 MB rope, and why tree shape does not matter

Large inputs go in as chunks so that every leaf is small. The fingerprint is a property of the bytes, not of the tree, so two ropes built with different chunk sizes over the same bytes have the same root fingerprint, and both equal the direct hash of the flat bytes.

In [4]:
import random, time
from hashrope import rope_height

def rope_from_chunks(data: bytes, chunk: int, h):
    root = None
    for i in range(0, len(data), chunk):
        leaf = Leaf(data[i:i + chunk], h)
        root = leaf if root is None else rope_concat(root, leaf, h)
    return root

random.seed(2026)
N = 1_000_000
data = bytes(random.getrandbits(8) for _ in range(N))

t0 = time.perf_counter(); r4k = rope_from_chunks(data, 4096, h); t1 = time.perf_counter()
print(f"4 KB leaves : built in {t1 - t0:.3f} s, height {rope_height(r4k)}, {rope_len(r4k):,} bytes")
t0 = time.perf_counter(); r1k = rope_from_chunks(data, 1000, h); t1 = time.perf_counter()
print(f"1 KB leaves : built in {t1 - t0:.3f} s, height {rope_height(r1k)}")
t0 = time.perf_counter(); direct = h.hash(data); t1 = time.perf_counter()
print(f"flat hash   : {t1 - t0:.3f} s")

print("same fingerprint for both shapes and the flat bytes:", rope_hash(r4k) == rope_hash(r1k) == direct)

4 KB leaves : built in 0.243 s, height 10, 1,000,000 bytes


1 KB leaves : built in 0.231 s, height 11
flat hash   : 0.213 s
same fingerprint for both shapes and the flat bytes: True


## 4. Edit without rehashing

Insert four bytes in the middle of the 1 MB rope: split at the position, join the pieces back with the new leaf in between, read the root. The unchanged subtrees are shared with the previous version (nothing is copied), only the O(log n) nodes on the path to the edit are rebuilt, and the root fingerprint is ready immediately. The naive alternative rebuilds the byte string and rehashes all of it.

The old root is still valid: every version stays addressable, which is what makes snapshots and branches free.

In [5]:
pos, patch = N // 2, b"DEV!"

t0 = time.perf_counter()
l, r = rope_split(r4k, pos, h)
edited = rope_concat(rope_concat(l, Leaf(patch, h), h), r, h)
rope_fp = rope_hash(edited)
t_rope = time.perf_counter() - t0

t0 = time.perf_counter()
naive_fp = h.hash(data[:pos] + patch + data[pos:])
t_naive = time.perf_counter() - t0

print(f"rope  edit + fingerprint : {t_rope * 1e6:8.0f} us")
print(f"naive rebuild + rehash   : {t_naive * 1e6:8.0f} us")
print(f"ratio                    : {t_naive / t_rope:8.0f} x")
print("fingerprints agree       :", rope_fp == naive_fp)
print("old version still intact :", rope_hash(r4k) == direct, "| new version:", rope_len(edited), "bytes")

cur = edited
samples = []
for _ in range(10):
    p = random.randrange(0, rope_len(cur))
    t0 = time.perf_counter()
    l, r = rope_split(cur, p, h)
    cur = rope_concat(rope_concat(l, Leaf(b"x", h), h), r, h)
    rope_hash(cur)
    samples.append((time.perf_counter() - t0) * 1e6)
print("ten more random inserts, us each:", [round(s) for s in samples], "| height still", rope_height(cur))

rope  edit + fingerprint :     1392 us
naive rebuild + rehash   :   219506 us
ratio                    :      158 x
fingerprints agree       : True
old version still intact : True | new version: 1000004 bytes
ten more random inserts, us each: [960, 963, 939, 960, 908, 1272, 881, 948, 899, 1231] | height still 10


## 5. Verify a copy with one comparison, then find where it differs

A service that receives a copy of the sequence can compare root fingerprints: one integer comparison. If they differ, `rope_lcp` returns the exact byte offset of the first difference by walking both trees. Shared subtrees are skipped; bytes that were stored independently are compared for real, so this is exact and never trusts a fingerprint.

In [6]:
from hashrope import rope_equal, rope_lcp

tampered = bytearray(data); tampered[123_456] ^= 1          # flip one bit somewhere in the middle
copy = rope_from_chunks(bytes(tampered), 4096, h)

print("root fingerprints equal:", rope_hash(copy) == rope_hash(r4k))
t0 = time.perf_counter(); where = rope_lcp(r4k, copy); t1 = time.perf_counter()
print(f"first differing byte at offset {where:,} (found in {(t1 - t0) * 1e3:.2f} ms)")

t0 = time.perf_counter(); same = rope_equal(r4k, r1k); t1 = time.perf_counter()
print(f"exact equality across different tree shapes: {same} ({(t1 - t0) * 1e3:.2f} ms)")

root fingerprints equal: False
first differing byte at offset 123,456 (found in 0.16 ms)
exact equality across different tree shapes: True (2.65 ms)


## 6. Repetition: a sequence larger than every disk on Earth, in one node

A `RepeatNode` stores a child and a count. Its fingerprint comes from a closed-form geometric sum, so q = 2^60 copies of a seven-byte pattern (about 8 exabytes if written out) get a length, a fingerprint and a mid-sequence substring fingerprint in microseconds.

In [7]:
from hashrope import rope_repeat

pattern = Leaf(b"devfest", h)
t0 = time.perf_counter()
big = rope_repeat(pattern, 1 << 60, h)
L = rope_len(big)
fp = rope_hash(big)
mid = rope_substr_hash(big, L // 2, 12, h)
t1 = time.perf_counter()

print(f"represented length : {L:,} bytes (about {L / 1e18:.1f} EB)")
print(f"fingerprint        : {hex(fp)}")
print(f"substring at middle: {hex(mid)}")
print(f"time for all three : {(t1 - t0) * 1e6:.0f} us")

expected = (b"devfest" * 4)[(L // 2) % 7:][:12]           # what those 12 bytes would be, if materialised
print("middle substring checks out against the materialised bytes:", mid == h.hash(expected))

represented length : 8,070,450,532,247,928,832 bytes (about 8.1 EB)
fingerprint        : 0x16099e895a235e2e
substring at middle: 0x1437c7888eb4ce2e
time for all three : 130 us
middle substring checks out against the materialised bytes: True


## 7. Fingerprints are candidates, not proof

A polynomial fingerprint is not a cryptographic digest. Under the default profile the byte strings `[0, 131]` and `[1, 0]` collide (131 = the base), and a determined adversary can craft collisions for a fixed base. Two rules follow:

1. When a decision depends on equality, confirm with `rope_equal` or `rope_lcp`. They compare bytes and never consult the fingerprint.
2. For adversarial settings (authentication, tamper evidence against an attacker) pair hashrope with a cryptographic hash such as SHA-256. hashrope is for cheap identity and change detection on data you control.

The `MERSENNE_127` profile gives a far smaller accidental collision probability when you want it.

In [8]:
from hashrope import MERSENNE_127

a = Leaf(bytes([0, 131, 42]), h)
b = Leaf(bytes([1, 0, 42]), h)
print("fingerprints equal :", rope_hash(a) == rope_hash(b))     # True: a known default-profile collision
print("bytes equal        :", rope_equal(a, b))                   # False: the exact check is not fooled
print("common prefix      :", rope_lcp(a, b), "bytes")

h127 = PolynomialHash(prime=MERSENNE_127)
print("wider profile      :", h127.profile)
print("collision survives?", rope_hash(Leaf(bytes([0, 131]), h127)) == rope_hash(Leaf(bytes([1, 0]), h127)), "(a wider prime does not fix a base-structured collision; exact APIs do)")

fingerprints equal : True
bytes equal        : False
common prefix      : 0 bytes
wider profile      : HashProfile(prime=170141183460469231731687303715884105727, base=131)
collision survives? True (a wider prime does not fix a base-structured collision; exact APIs do)


## 8. Agent memory: one context, many branches, shared prefix

Treat an agent's context as a rope of turns. A snapshot is just keeping the old root. A branch is a concat of a new continuation onto the shared prefix. The two branches share the prefix's nodes in memory, the prefix fingerprint can be read from either branch without rehashing, and `rope_lcp` proves exactly how much of the two contexts is identical (useful for prefix-cache routing between inference workers).

In [9]:
turns = b"system: you are a travel planning agent\nuser: plan a two day trip to Orlando\n" * 1000
prefix = rope_from_chunks(turns, 512, h)

branch_a = rope_concat(prefix, Leaf(b"A: day one at the parks, day two at the springs", h), h)
branch_b = rope_concat(prefix, Leaf(b"B: day one at the springs, day two at the parks", h), h)

def node_ids(node, seen):
    if node is None or id(node) in seen:
        return seen
    seen.add(id(node))
    for attr in ("left", "right", "child"):
        node_ids(getattr(node, attr, None), seen)
    return seen

ids_a, ids_b = node_ids(branch_a, set()), node_ids(branch_b, set())
print(f"nodes in branch A: {len(ids_a)}, in branch B: {len(ids_b)}, shared: {len(ids_a & ids_b)}, total distinct: {len(ids_a | ids_b)}")
print("shared prefix length proven by rope_lcp:", rope_lcp(branch_a, branch_b), "of", rope_len(prefix), "prefix bytes")
print("prefix fingerprint read from branch A  :", rope_substr_hash(branch_a, 0, rope_len(prefix), h) == rope_hash(prefix))
print("prefix fingerprint read from branch B  :", rope_substr_hash(branch_b, 0, rope_len(prefix), h) == rope_hash(prefix))

nodes in branch A: 303, in branch B: 303, shared: 292, total distinct: 314
shared prefix length proven by rope_lcp: 77000 of 77000 prefix bytes
prefix fingerprint read from branch A  : True
prefix fingerprint read from branch B  : True


## 9. The same fingerprint from JavaScript

Colab ships Node.js, so the npm package runs here too. Same algorithm, same defaults, same fixtures in every port: a fingerprint computed in a Python pipeline matches the one computed in a Node service or a browser. The script below prints the hello world fingerprint, the 2^60 repeat fingerprints, the collision pair and the 1 MB edit timing, all from the npm package.

In [10]:
!node --version
!npm install --silent --no-audit --no-fund hashrope@0.3.4

v22.23.4


In [11]:
%%writefile demo.mjs
import { PolynomialHash, Leaf, ropeConcat, ropeSplit, ropeHash, ropeLen, ropeSubstrHash,
         ropeRepeat, ropeEqual, ropeLcp, VERSION } from 'hashrope';

const enc = (s) => new TextEncoder().encode(s);
const hex = (n) => '0x' + n.toString(16);
const h = new PolynomialHash();
console.log('hashrope (npm)', VERSION);

// 1. the same fingerprint Python printed
const hw = ropeConcat(new Leaf(enc('hello '), h), new Leaf(enc('world'), h), h);
console.log(`hello world         : ${hex(ropeHash(hw))} ${ropeHash(hw) === h.hash(enc('hello world')) ? '(matches direct hash)' : '(MISMATCH)'}`);

// 2. the 2^60 repeat, same two numbers as Python
const big = ropeRepeat(new Leaf(enc('devfest'), h), 1n << 60n, h);
const L = ropeLen(big);
console.log(`repeat fingerprint  : ${hex(ropeHash(big))}`);
console.log(`substring at middle : ${hex(ropeSubstrHash(big, L / 2n, 12n, h))}`);

// 3. the collision pair, caught by the exact API
const c1 = new Leaf(new Uint8Array([0, 131, 42]), h), c2 = new Leaf(new Uint8Array([1, 0, 42]), h);
console.log(`collision pair      : fingerprints equal ${ropeHash(c1) === ropeHash(c2)} | bytes equal ${ropeEqual(c1, c2)} | lcp ${ropeLcp(c1, c2)}`);

// 4. 1 MB edit versus naive rehash
const N = 1_000_000, data = new Uint8Array(N);
let s = 2026; for (let i = 0; i < N; i++) { s = (s * 1103515245 + 12345) & 0x7fffffff; data[i] = (s >> 16) & 255; }
let root = null;
for (let i = 0; i < N; i += 4096) { const leaf = new Leaf(data.subarray(i, i + 4096), h); root = root === null ? leaf : ropeConcat(root, leaf, h); }
const patch = enc('DEV!');
let t0 = performance.now();
const [a, b] = ropeSplit(root, BigInt(N / 2), h);
const edited = ropeConcat(ropeConcat(a, new Leaf(patch, h), h), b, h);
const fp = ropeHash(edited);
const tRope = performance.now() - t0;
const flat = new Uint8Array(N + 4); flat.set(data.subarray(0, N / 2)); flat.set(patch, N / 2); flat.set(data.subarray(N / 2), N / 2 + 4);
t0 = performance.now(); const naive = h.hash(flat); const tNaive = performance.now() - t0;
console.log(`1 MB edit + fingerprint: rope ${tRope.toFixed(2)} ms | naive rehash ${tNaive.toFixed(1)} ms | agree ${fp === naive}`);

Writing demo.mjs


In [12]:
!node demo.mjs

hashrope (npm) 0.3.4
hello world         : 0x5f87bb7db23cc7b (matches direct hash)
repeat fingerprint  : 0x16099e895a235e2e
substring at middle : 0x1437c7888eb4ce2e
collision pair      : fingerprints equal true | bytes equal false | lcp 0


1 MB edit + fingerprint: rope 0.85 ms | naive rehash 119.4 ms | agree true


## 10. Rust and Java

The same operations, same defaults, same fingerprints. Not executed here (Colab has no Rust or Java toolchain by default), shown for reference.

Rust (`cargo add hashrope`), arena-backed and `no_std + alloc`:

```rust
use hashrope::Arena;

let mut a = Arena::new();
let hello = a.from_bytes(b"hello ");
let world = a.from_bytes(b"world");
let hw = a.concat(hello, world);
assert_eq!(a.hash(hw), a.hash_bytes(b"hello world"));   // 0x5f87bb7db23cc7b
let (left, _right) = a.split(hw, 6);
assert_eq!(a.substr_hash(hw, 6, 5), a.hash_bytes(b"world"));
let again = a.from_bytes(b"hello ");
assert!(a.equal(left, again));                          // exact, never a fingerprint
```

Java (`io.github.jemsbhai:hashrope:0.3.4` on Maven Central), Java 17+:

```java
import hashrope.*;
import static java.nio.charset.StandardCharsets.UTF_8;

PolynomialHash h = new PolynomialHash();
Node hw = Rope.concat(Rope.fromBytes("hello ".getBytes(UTF_8), h),
                      Rope.fromBytes("world".getBytes(UTF_8), h), h);
Rope.hash(hw).equals(h.hash("hello world".getBytes(UTF_8)));   // true
Rope.Split halves = Rope.split(hw, 6, h);
Rope.equal(halves.left(), Rope.fromBytes("hello ".getBytes(UTF_8), h));   // true, exact
```

| Python | JavaScript / TypeScript | Rust | Java |
|---|---|---|---|
| `rope_concat` | `ropeConcat` | `arena.concat` | `Rope.concat` |
| `rope_split` | `ropeSplit` | `arena.split` | `Rope.split` |
| `rope_substr_hash` | `ropeSubstrHash` | `arena.substr_hash` | `Rope.substrHash` |
| `rope_repeat` | `ropeRepeat` | `arena.repeat` | `Rope.repeat` |
| `rope_equal` / `rope_lcp` | `ropeEqual` / `ropeLcp` | `arena.equal` / `arena.lcp` | `Rope.equal` / `Rope.lcp` |
| `int` | `bigint` | `u64` (or `BigUint` with the bigint feature) | `BigInteger` |

## 11. Where to go next

- Poke the live tree in your browser: https://hashrope.lovable.app/
- Cheat sheet (install lines, the five calls, the two rules): https://gist.github.com/jemsbhai (file hashrope-cheatsheet.md)
- Source, release records and every benchmark, including the ones hashrope loses: https://github.com/jemsbhai/hashrope
- The ICTAI 2026 paper record: https://github.com/jemsbhai/hashrope-paper
- Genomics variant (tandem repeats as repeat nodes): `pip install hashrope-bio`, `cargo add hashrope-bio`

Two rules to carry into your next project: a fingerprint is a candidate match, confirm with the exact APIs when a decision depends on it; and hashrope is not a cryptographic digest, so pair it with SHA-256 when an attacker is in the threat model.

## 12. Challenges

Three exercises, using only operations from the sections above. Each code cell sets up the problem and ends with a check that prints PASS when your answer is right. The checks never print the answer itself.

If you finished the hands-on early, start here.

### Challenge 1: find every divergence

Section 5 found the *first* differing byte with one `rope_lcp`. This copy of a 256 KB sequence was tampered with in **three** places. Find all three offsets.

Hint: `rope_range_lcp(a, start, b, start, n)` returns the length of the common prefix of the two ranges starting at `start`. After a difference at position `p`, continue the comparison from `p + 1`.

In [ ]:
import random
from hashrope import rope_range_lcp

rng = random.Random(20261120)
base = bytes(rng.randrange(256) for _ in range(256 * 1024))
orig = rope_from_chunks(base, 4096, h)

tampered = bytearray(base)
_truth1 = sorted(rng.sample(range(1000, len(base) - 1000), 3))
for p in _truth1:
    tampered[p] ^= 0xFF
copy1 = rope_from_chunks(bytes(tampered), 4096, h)

diffs = []  # your code here: fill with the three offsets, ascending

print("PASS" if diffs == _truth1 else f"not yet: you found {diffs}")

### Challenge 2: compose a cache key without the parts

A bundle is `header || payload`. A cache already stores the fingerprint of each part and the payload's length, but not the bytes. Compose the fingerprint of the whole bundle from those three numbers alone, then check it against the real rope.

This is `hash_concat` from the recipes: the key of A followed by B, from the keys of the parts, with no re-read of either part.

In [ ]:
header = b"HDR:" + bytes(range(64))
payload = bytes((i * 7) % 256 for i in range(10_000))
ra, rb = rope_from_chunks(header, 4096, h), rope_from_chunks(payload, 4096, h)

fp_a, len_b, fp_b = rope_hash(ra), rope_len(rb), rope_hash(rb)

key_ab = None  # your code here: compose it from fp_a, len_b, fp_b only

_truth2 = rope_hash(rope_concat(ra, rb, h))
print("PASS" if key_ab == _truth2 else "not yet: that is not the bundle's fingerprint")

### Challenge 3: find the repeated window

A 32 KB stream contains a 2 KB chunk that appears **twice**. You are given the fingerprint of the chunk. Find the offset of its second occurrence, using window fingerprints (`rope_substr_hash`) rather than comparing bytes.

The first occurrence is at offset 6,000. Windows can start at any byte offset, so scan.

In [ ]:
rng3 = random.Random(42)
stream = bytearray(rng3.randrange(256) for _ in range(32 * 1024))
stream[20_000:22_000] = stream[6_000:8_000]          # the same 2 KB, again
srope = rope_from_chunks(bytes(stream), 4096, h)

target = rope_substr_hash(srope, 6_000, 2_000, h)    # fingerprint of the chunk

second = None  # your code here: the offset of the second occurrence

print("PASS" if second == 20_000 else f"not yet: {second}")

Done all three? You have used the whole argument of the talk: fingerprints to find candidates cheaply, exact comparisons to decide, and composition to avoid re-reading data.

Stuck on one: re-read section 2 (substring fingerprints), section 5 (verify and locate), and the cache-key recipe in the talk's cheat sheet.